# 03 · Sales Trends & Time-Series Analysis

**Goal:** Understand how sales and profit evolve over time — monthly, quarterly, and yearly.

---
### Questions
1. What are the monthly and quarterly sales/profit trends?
2. Are there seasonal patterns in sales?
3. Which year-over-year growth trends exist by category?
4. How does shipping lag vary by ship mode?

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from src.visualizations import plot_monthly_trend, plotly_monthly_trend

%matplotlib inline
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('../data/superstore_clean.csv', parse_dates=['Order Date', 'Ship Date'])
print(f'Loaded: {df.shape}')

## Q1 · Monthly Sales & Profit Trend

In [ ]:
plot_monthly_trend(df, value_col='Sales', title='Monthly Sales Trend')
plot_monthly_trend(df, value_col='Profit', title='Monthly Profit Trend')

In [ ]:
# Interactive version
fig = plotly_monthly_trend(df, value_cols=['Sales', 'Profit'],
                            title='Monthly Sales & Profit (Interactive)')
fig.show()

## Q2 · Quarterly Sales

In [ ]:
quarterly = df.groupby('Order Quarter')[['Sales', 'Profit']].sum().reset_index()
quarterly = quarterly.sort_values('Order Quarter')

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(quarterly['Order Quarter'], quarterly['Sales'], color='steelblue', alpha=0.7, label='Sales')
ax2 = ax.twinx()
ax2.plot(quarterly['Order Quarter'], quarterly['Profit'], color='orange',
         marker='o', linewidth=2, label='Profit')
ax.set_xticklabels(quarterly['Order Quarter'], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Quarterly Sales (bars) & Profit (line)')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Q3 · Seasonality — Month-of-Year Pattern

In [ ]:
import calendar

monthly_avg = df.groupby('Order Month')['Sales'].mean().reset_index()
monthly_avg['Month Name'] = monthly_avg['Order Month'].apply(lambda m: calendar.month_abbr[m])

plt.figure(figsize=(10, 5))
sns.barplot(data=monthly_avg, x='Month Name', y='Sales', palette='Blues_d')
plt.title('Average Sales by Month (Seasonality)')
plt.ylabel('Avg Sales ($)')
plt.tight_layout()
plt.show()

## Q4 · Year-over-Year Growth by Category

In [ ]:
yoy = df.groupby(['Order Year', 'Category'])['Sales'].sum().unstack('Category').fillna(0)
yoy_pct = yoy.pct_change() * 100

print('Year-over-Year Sales Growth (%):')
print(yoy_pct.round(1).to_string())

yoy.plot(kind='bar', figsize=(9, 5), colormap='Set2', rot=0,
         title='Annual Sales by Category')
plt.ylabel('Total Sales ($)')
plt.tight_layout()
plt.show()

## Q5 · Shipping Lag by Ship Mode

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Ship Mode', y='Ship Lag (days)', palette='Set3')
plt.title('Shipping Lag Distribution by Ship Mode')
plt.ylabel('Days from Order to Ship')
plt.tight_layout()
plt.show()

print(df.groupby('Ship Mode')['Ship Lag (days)'].describe().round(1).to_string())